# 04 · Recommendations Engine

> **Parte 3 do desafio:** um sistema que analisa uma campanha e gera pelo menos três recomendações de melhoria, acionáveis, quantificadas, contextuais e priorizadas por impacto estimado na performance.

Este notebook registra o desenho do engine e calcula as evidências que ele usa. O engine e a avaliação nas 500 campanhas ainda não estão aqui.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from klike.analysis import CONTEXT_WITHIN_PLATFORM, fit_model
from klike.config import TARGET
from klike.data import load_clean
from klike.evidence import (
    PERFORMANCE_METRICS,
    PERFORMANCE_PATH,
    SCORE_PATH,
    TRANSITIONS,
    performance_data,
    performance_evidence,
    save_evidence,
    score_differences,
    transition_effect,
)
from klike.formatting import fmt_number, fmt_pct, fmt_signed, fmt_signed_pct

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 140)
PLATFORMS = ["Meta", "TikTok", "LinkedIn"]
BY_KEY = {transition.key: transition for transition in TRANSITIONS}
cleaned = load_clean()
data = performance_data(cleaned)

## 1. O que o engine precisa fazer

| Requisito do enunciado | Como o engine responde |
| :-- | :-- |
| Recebe uma campanha | Os 13 atributos de pré-publicação bastam, como no modelo da Parte 2. Métricas observadas de uma campanha histórica entram só como contexto, rotuladas como observadas. |
| Pelo menos 3 recomendações | Todas as melhorias defensáveis, em classes de evidência. Quando não há três, a saída diz isso, sem completar a lista com sugestões artificiais (seção 3.5). |
| Acionáveis e quantificadas | Cada recomendação é uma edição concreta do vídeo, com a diferença de score prevista e a associação histórica com a métrica do objetivo, cada uma com seu intervalo. |
| Contextuais | O contexto define o que se aplica à campanha, a métrica que importa, o efeito do formato em cada plataforma e o suporte histórico. Não inventa efeitos diferentes por segmento que a base não mostra. |
| Priorizadas por impacto | Primeiro a classe de evidência, depois o impacto estimado na métrica do objetivo. Prioridade e impacto aparecem separados. |

Três coisas que o engine não faz, de propósito:

- **Não muda o contexto da campanha.** Plataforma, categoria, objetivo, público e retargeting não são edições do vídeo.
- **Não trata valor ausente como ausência da característica.** Uma legenda sem marcação não vira "sem legenda".
- **Não soma efeitos de edições combinadas.** Cada recomendação é avaliada sozinha.

## 2. Três camadas que não se misturam

| Camada | Pergunta | Fonte | Incerteza |
| :-- | :-- | :-- | :-- |
| Diferença prevista de score | Quanto o modelo final prevê de diferença ao editar só esta característica? | `KlikeScorePredictor`, a Ridge com a interação plataforma × formato, nas 500 campanhas | Bootstrap de todo o treino do modelo congelado. Não é o MAE, que mede o erro de prever o score de uma campanha. |
| Associação histórica com performance | Quanto a métrica do objetivo difere entre campanhas parecidas com e sem a característica? | O contraste exato da transição nas regressões da Parte 1 | Intervalo de 95%, com erros robustos |
| Qualidade da evidência | O quanto essa associação resiste à multiplicidade e às escolhas de análise? | Benjamini-Hochberg na família de 70 contrastes e duas sensibilidades | Classes robusta, indício e inconclusiva |

Ao lado delas, o **suporte histórico** conta quantas campanhas parecidas existem de cada lado da comparação. É informação de contexto e de risco de extrapolação, não de evidência estatística.

Nenhuma das camadas mede efeito causal. A diferença de score é o que o modelo prevê; a associação compara campanhas parecidas no que foi medido. Uma classe de evidência não é uma probabilidade de sucesso.

## 3. Protocolo das evidências

As escolhas abaixo foram fixadas antes dos números das seções 4 a 7. Mas foram feitas depois da EDA e de um diagnóstico exploratório que já tinha visto essas estimativas. Por isso, esta parte é exploratória: a correção controla a multiplicidade dentro da família, mas não torna as conclusões confirmatórias.

### 3.1 A família de 70 hipóteses

Cada transição é uma edição concreta do vídeo, medida pelo contraste exato no modelo: texto alto → médio é a diferença entre os coeficientes de médio e de alto, e não o efeito de "texto médio contra baixo". A família inclui todas as transições avaliadas, inclusive as que o engine descarta, como duração, música/voz e formato quadrado na Meta. Tirar da correção o que foi olhado e descartado a tornaria otimista.

In [2]:
family = pd.DataFrame(
    [
        {
            "transição": t.label,
            "campo": t.attribute,
            "estimada em": (
                "todas as campanhas" if t.platform is None else f"campanhas do {t.platform}"
            ),
            "termo do modelo": t.contrast,
        }
        for t in TRANSITIONS
    ]
)
display(family.set_index("transição"))
print(
    f"{len(TRANSITIONS)} transições × {len(PERFORMANCE_METRICS)} métricas "
    f"({', '.join(PERFORMANCE_METRICS.values())}) = {len(TRANSITIONS) * len(PERFORMANCE_METRICS)} "
    "hipóteses"
)

,campo,estimada em,termo do modelo
transição,,,
hook: não → sim,has_hook,todas as campanhas,has_hook
rosto humano: não → sim,has_face,todas as campanhas,has_face
CTA: não → sim,has_cta,todas as campanhas,has_cta
legenda: não → sim,has_subtitle,todas as campanhas,subtitle_status[T.sim]
texto: alto → médio,text_density,todas as campanhas,text_density[T.medium] - text_density[T.high]
texto: alto → baixo,text_density,todas as campanhas,-text_density[T.high]
texto: baixo → médio,text_density,todas as campanhas,text_density[T.medium]
"música/voz: +0,1 (de 0,4 para 0,5)",music_voice_ratio,todas as campanhas,musica_decimos
duração: dobrar (de 15 para 30 s),video_duration_s,todas as campanhas,np.log(video_duration_s)


14 transições × 5 métricas (CTR, engajamento, retenção, conversões por mil impressões, ROAS) = 70 hipóteses


### 3.2 Modelo principal, correção e classes

- **Modelo principal:** o mesmo da Parte 1. A métrica entra em log, com erros robustos (HC3), controlando plataforma, categoria, objetivo, faixa etária, retargeting e os outros atributos do criativo. O formato é estimado dentro de cada plataforma, porque o que um formato significa muda de uma para outra, e porque o TikTok não tem vídeo horizontal.
- **Correção:** Benjamini-Hochberg sobre as 70 hipóteses de uma vez.
- **Classes:** robusta (q < 0,05), indício (p < 0,05, mas q ≥ 0,05) e inconclusiva (p ≥ 0,05). Inconclusiva não quer dizer que o efeito não existe: quer dizer que esta base não permite afirmar.

### 3.3 Sensibilidades

O modelo principal define a classe. As sensibilidades só dizem se ela é estável:

- **Sem as correções por plausibilidade.** Em 5 campanhas, a Parte 1 corrigiu um registro escolhendo o valor mais plausível, sem uma segunda conta para confirmar. A família inteira é refeita sem elas.
- **Binomial negativa para conversões.** A regressão em log descarta as 19 campanhas sem nenhuma conversão; a binomial negativa as mantém. As linhas de conversões são refeitas com ela, e a família é reclassificada.
- **Limitações que não viram reestimativa:** os possíveis tetos de registro do CTR, do engajamento e da retenção, e o ROAS que não se aplica a Branding. Ficam anotados em cada linha.

Um resultado é **estável** quando a classe não muda em nenhuma sensibilidade aplicável. A mudança pode ir nos dois sentidos, e as duas importam: uma classe que cai mostra fragilidade, e uma que sobe mostra que a conclusão depende do modelo.

### 3.4 A métrica de cada objetivo

| Objetivo | Métrica | Ressalva |
| :-- | :-- | :-- |
| traffic | CTR | Possível teto de registro em 15%, que pode subestimar efeitos em campanhas que já clicam muito |
| engagement | Engajamento | 32 campanhas sem registro e possível teto em 10% |
| awareness | Retenção (tempo assistido sobre a duração) | É atenção, não lembrança de marca; 32 campanhas estão em 100%. Não há alcance nem frequência na base. |
| conversions | Conversões por mil impressões | Combina clique e conversão. A conversão por clique não mostra efeito de criativo, e o ROAS só se associa ao texto, de forma sensível; ele aparece como informação secundária. |
| app_install | Conversões por mil impressões | A base não distingue instalações de outras conversões |

O engine permite trocar a métrica por parâmetro, porque o mapeamento é uma premissa de produto, não um resultado.

### 3.5 Classes de recomendação e prioridade

A classe de cada recomendação vem da associação com a métrica do objetivo da campanha:

| Classe | Critério | Como é comunicada |
| :-- | :-- | :-- |
| A | Associação favorável robusta e estável | Melhoria com evidência de performance nesta base |
| B | Indício favorável, ou robusta mas sensível | Melhoria com evidência fraca ou dependente da análise |
| C | Associação inconclusiva, com aumento de score previsto cujo intervalo fica acima de zero | Oportunidade de score sem evidência de performance: hipótese para teste A/B |

- **Exclusões:** associação desfavorável com p < 0,05 na métrica do objetivo, mesmo que o score suba; transições descartadas na seção 8; formato de destino com menos de 10 campanhas na plataforma.
- **Sinal desfavorável, mas inconclusivo:** a recomendação fica em C, com aviso explícito.
- **Prioridade:** A antes de B, B antes de C. Dentro de A e de B, ordena a estimativa na métrica do objetivo; dentro de C, a diferença de score. Prioridade não é impacto: uma recomendação B pode ter impacto estimado maior que uma A e ficar atrás, porque a evidência dela é mais fraca.
- **Mínimo de três:** o engine lista todas as recomendações elegíveis e informa quando há menos de três. Não conta uma edição mais de uma vez, nem preenche a lista com "mantenha".

### 3.6 Suporte histórico

Para cada recomendação, o engine conta as campanhas com o valor atual e com o valor sugerido, em recortes que vão do mais específico ao mais amplo: plataforma, categoria e objetivo; plataforma e objetivo; plataforma; todas. Usa o primeiro recorte com pelo menos 10 campanhas de cada lado e registra qual foi.

- O mínimo de 10 é uma decisão operacional, a mesma das interações da Parte 2. A avaliação vai testar 5 e 20.
- Suporte baixo vira aviso de risco de extrapolação. Ele nunca muda a classe de evidência de um efeito geral.
- Suporte alto não é probabilidade de sucesso: diz quantas campanhas parecidas existem, não se a mudança vai funcionar.

## 4. Associação com performance

Uma linha por transição, uma coluna por métrica: a variação estimada ao fazer a edição, com o intervalo de 95% e a classe quando não é robusta.

In [3]:
evidence = performance_evidence(cleaned)
LEVEL_SUFFIX = {"robusto": "", "indício": " (indício)", "inconclusivo": " (inconclusivo)"}


def effect_text(row: pd.Series) -> str:
    interval = f"[{fmt_signed_pct(row['ic_inferior'])}, {fmt_signed_pct(row['ic_superior'])}]"
    return f"{fmt_signed_pct(row['efeito'])} {interval}{LEVEL_SUFFIX[row['evidencia']]}"


display(
    evidence.assign(texto=evidence.apply(effect_text, axis=1))
    .pivot(index="descricao", columns="metrica_nome", values="texto")
    .loc[[t.label for t in TRANSITIONS], list(PERFORMANCE_METRICS.values())]
    .rename_axis(index="transição", columns=None)
)
levels = evidence["evidencia"].value_counts()
print(
    f"{len(evidence)} hipóteses | robustas: {levels.get('robusto', 0)} | indícios: "
    f"{levels.get('indício', 0)} | inconclusivas: {levels.get('inconclusivo', 0)}"
)

,CTR,engajamento,retenção,conversões por mil impressões,ROAS
transição,,,,,
hook: não → sim,"+51% [+31%, +75%]","+38% [+16%, +65%]","+39% [+26%, +53%]","+48% [+26%, +75%]","−4% [−17%, +10%] (inconclusivo)"
rosto humano: não → sim,"+23% [+7%, +42%]","+43% [+20%, +69%]","+11% [0%, +23%] (indício)","+29% [+10%, +52%]","+2% [−11%, +18%] (inconclusivo)"
CTA: não → sim,"+9% [−6%, +27%] (inconclusivo)","+4% [−15%, +26%] (inconclusivo)","−3% [−13%, +8%] (inconclusivo)","+8% [−9%, +28%] (inconclusivo)","+5% [−9%, +21%] (inconclusivo)"
legenda: não → sim,"+11% [−4%, +28%] (inconclusivo)","+6% [−12%, +27%] (inconclusivo)","+8% [−3%, +19%] (inconclusivo)","+14% [−4%, +35%] (inconclusivo)","+5% [−9%, +20%] (inconclusivo)"
texto: alto → médio,"+23% [+2%, +49%] (indício)","+8% [−14%, +35%] (inconclusivo)","+29% [+13%, +48%]","+36% [+8%, +71%]","+21% [+1%, +45%] (indício)"
texto: alto → baixo,"+18% [−4%, +45%] (inconclusivo)","+8% [−15%, +37%] (inconclusivo)","+23% [+7%, +41%]","+31% [+3%, +66%] (indício)","+32% [+8%, +62%]"
texto: baixo → médio,"+4% [−11%, +23%] (inconclusivo)","0% [−18%, +21%] (inconclusivo)","+5% [−6%, +17%] (inconclusivo)","+4% [−14%, +25%] (inconclusivo)","−9% [−21%, +6%] (inconclusivo)"
"música/voz: +0,1 (de 0,4 para 0,5)","+1% [−3%, +5%] (inconclusivo)","+3% [−2%, +8%] (inconclusivo)","+4% [+2%, +6%]","+1% [−3%, +5%] (inconclusivo)","−1% [−4%, +3%] (inconclusivo)"
duração: dobrar (de 15 para 30 s),"+1% [−8%, +10%] (inconclusivo)","−3% [−12%, +8%] (inconclusivo)","0% [−6%, +6%] (inconclusivo)","+2% [−8%, +14%] (inconclusivo)","−2% [−10%, +7%] (inconclusivo)"


70 hipóteses | robustas: 13 | indícios: 4 | inconclusivas: 53


- **Hook e rosto têm as associações mais fortes e mais estáveis.** Hook: CTR +51%, engajamento +38%, retenção +39% e conversões por mil +48%, todas robustas. Rosto: CTR +23%, engajamento +43% e conversões por mil +29%, robustas; retenção, só indício. Nenhum dos dois se associa ao ROAS.
- **Sair do texto alto é a terceira alavanca.** Para o médio, a retenção é 29% maior, com evidência robusta; conversões por mil (+36%), CTR (+23%) e ROAS (+21%) ficam mais fracos ou sensíveis (seção 5). Para o baixo, a retenção é 23% maior.
- **Legenda, CTA e texto baixo → médio não mostram associação com performance.** Todos os 15 contrastes são inconclusivos. É o padrão que a Parte 1 já tinha visto: o score dá pontos a essas características, mas a performance não acompanha com clareza nesta base.
- **Música/voz:** +0,1 de música se associa a 3,9% a mais de retenção, robusto, mas pequeno, e a nada nas outras métricas.
- **Duração não mostra associação com nada,** com intervalos estreitos: dobrar a duração fica entre −8% e +10% no CTR e entre −6% e +6% na retenção.
- **Formato só tem evidência na Meta, do horizontal para o vertical,** e só no engajamento, com +63%. Os demais formatos, inclusive no TikTok e no LinkedIn, são inconclusivos, com intervalos largos.

## 5. Sensibilidades

As linhas em que a classe muda em alguma sensibilidade:

In [4]:
unstable = evidence[~evidence["estavel"]]
display(
    unstable.assign(
        principal=unstable.apply(effect_text, axis=1),
        sem_plausibilidade=[
            f"{fmt_signed_pct(e)} ({level})"
            for e, level in zip(
                unstable["efeito_sem_plausibilidade"],
                unstable["evidencia_sem_plausibilidade"],
                strict=True,
            )
        ],
        binomial_negativa=[
            f"{fmt_signed_pct(e)} ({level})" if pd.notna(e) else level
            for e, level in zip(
                unstable["efeito_binomial_negativa"],
                unstable["evidencia_binomial_negativa"],
                strict=True,
            )
        ],
    )
    .set_index(["descricao", "metrica_nome"])
    .loc[:, ["principal", "sem_plausibilidade", "binomial_negativa"]]
    .rename_axis(["transição", "métrica"])
)
print(
    f"Estáveis: {int(evidence['estavel'].sum())} de {len(evidence)} | sem estimativa na binomial "
    f"negativa: {(evidence['evidencia_binomial_negativa'] == 'não estimável').sum()} linhas"
)
# Estabilidade é da classe: a estimativa pode mudar de tamanho e, perto de zero, de sinal.
estimates = evidence[["efeito", "efeito_sem_plausibilidade", "efeito_binomial_negativa"]]
sensitivity_signs = np.sign(estimates.iloc[:, 1:])
flipped = (
    sensitivity_signs.ne(np.sign(estimates["efeito"]), axis=0) & sensitivity_signs.notna()
).any(axis=1)
print(
    f"Sinal invertido em alguma sensibilidade: {int(flipped.sum())} contrastes, de classe "
    f"{', '.join(sorted(evidence.loc[flipped, 'evidencia'].unique()))}; maior estimativa entre "
    f"eles, em módulo: {fmt_pct(estimates[flipped].abs().max().max(), 1)}"
)

principal  \
transição           métrica                                                          
legenda: não → sim  conversões por mil impressões  +14% [−4%, +35%] (inconclusivo)   
texto: alto → médio conversões por mil impressões                 +36% [+8%, +71%]   
                    ROAS                                +21% [+1%, +45%] (indício)   
texto: alto → baixo conversões por mil impressões       +31% [+3%, +66%] (indício)   
                    ROAS                                          +32% [+8%, +62%]   

                                                    sem_plausibilidade  \
transição           métrica                                              
legenda: não → sim  conversões por mil impressões  +16% (inconclusivo)   
texto: alto → médio conversões por mil impressões       +30% (indício)   
                    ROAS                           +17% (inconclusivo)   
texto: alto → baixo conversões por mil impressões       +26% (indício)   
                    ROAS                                +29% (indício)   

                                                  binomial_negativa  
transição           métrica                                          
legenda: não → sim  conversões por mil impressões    +23% (robusto)  
texto: alto → médio conversões por mil impressões    +40% (robusto)  
                    ROAS                              não se aplica  
texto: alto → baixo conversões por mil impressões    +31% (robusto)  
                    ROAS                              não se aplica

Estáveis: 65 de 70 | sem estimativa na binomial negativa: 2 linhas
Sinal invertido em alguma sensibilidade: 4 contrastes, de classe inconclusivo; maior estimativa entre eles, em módulo: 1,1%


- **O texto depende das correções por plausibilidade.** Sem as 5 campanhas corrigidas por plausibilidade, conversões por mil de alto → médio cai de robusta para indício, e o ROAS cai de indício para inconclusivo (de alto → baixo, de robusto para indício). A retenção, a evidência mais forte do texto, não muda. Na direção oposta, a binomial negativa deixa robustas as conversões de alto → baixo, que no log são só indício.
- **Log e binomial negativa discordam na legenda.** Em conversões, a regressão em log é inconclusiva (+14%), e a binomial negativa encontra +23%, robusta. Os dois modelos medem coisas diferentes: o log dá o mesmo peso a cada campanha e descarta as 19 sem conversão; a binomial negativa as mantém e dá mais peso a quem tem mais volume. Como o modelo principal é inconclusivo, a legenda continua sem evidência de performance, mas a saída do engine registra que essa conclusão depende do modelo.
- **No LinkedIn, a binomial negativa não converge** para o formato. Com 83 campanhas, o otimizador declarava convergência com parâmetros vazios; a função da Parte 1 passou a rejeitar esse caso, e as duas linhas ficam como não estimáveis.
- **Hook e rosto são estáveis em todas as sensibilidades.**
- **Estável quer dizer que a classe não muda, não que a estimativa fique igual.** As magnitudes variam entre as análises: as conversões de texto alto → médio vão de +36% para +30% sem as correções por plausibilidade. Em 4 contrastes inconclusivos, o sinal da estimativa chega a se inverter, sempre com valores de até 1,1% em módulo: são estimativas praticamente nulas, cujo sinal não carrega informação.

## 6. Contexto: o que muda de uma plataforma para outra

O enunciado lembra que o que funciona no TikTok pode não funcionar no LinkedIn. A Parte 1 testou isso com interações e não encontrou diferença robusta, o que não prova que os efeitos sejam iguais. Aqui, o mesmo contraste é estimado dentro de cada plataforma, e a variância entre plataformas é medida pelo método de DerSimonian-Laird. Ela diria quanto encolher cada estimativa em direção ao efeito geral.

In [5]:
OBJECTIVE_METRICS = {
    metric: name for metric, name in PERFORMANCE_METRICS.items() if metric != "roas_non_branding"
}


def between_platform_variance(estimates: np.ndarray, variances: np.ndarray):
    """Variância entre plataformas (DerSimonian-Laird) e o peso de cada uma no encolhimento."""
    weights = 1 / variances
    pooled = (weights * estimates).sum() / weights.sum()
    q = (weights * (estimates - pooled) ** 2).sum()
    scale = weights.sum() - (weights**2).sum() / weights.sum()
    tau2 = max(0.0, (q - (len(estimates) - 1)) / scale)
    return tau2, tau2 / (tau2 + variances)


rows = []
general = evidence.set_index(["transicao", "metrica"])["efeito"]
for key in ["hook", "rosto", "texto_alto_medio"]:
    transition = BY_KEY[key]
    for metric, name in OBJECTIVE_METRICS.items():
        estimates, variances, texts = [], [], {}
        for platform in PLATFORMS:
            fit = fit_model(
                data[data["platform"] == platform], metric, controls=CONTEXT_WITHIN_PLATFORM
            )
            test = fit.t_test(f"{transition.contrast} = 0")
            estimate, sd = float(np.squeeze(test.effect)), float(np.squeeze(test.sd))
            estimates.append(estimate)
            variances.append(sd**2)
            low, high = np.expm1([estimate - 1.96 * sd, estimate + 1.96 * sd])
            texts[platform] = (
                f"{fmt_signed_pct(np.expm1(estimate))} "
                f"[{fmt_signed_pct(low)}, {fmt_signed_pct(high)}]"
            )
        tau2, shrink = between_platform_variance(np.array(estimates), np.array(variances))
        rows.append(
            {
                "transição": transition.label,
                "métrica": name,
                "geral": fmt_signed_pct(general[(key, metric)]),
                **texts,
                "variância entre plataformas": fmt_number(tau2, 4),
                "peso da própria plataforma": " / ".join(fmt_number(s, 2) for s in shrink),
            }
        )
heterogeneity = pd.DataFrame(rows).set_index(["transição", "métrica"])
display(heterogeneity)
zero = (heterogeneity["variância entre plataformas"] == fmt_number(0, 4)).sum()
print(f"Variância entre plataformas igual a zero em {zero} de {len(heterogeneity)} combinações")

geral  \
transição               métrica                               
hook: não → sim         CTR                            +51%   
                        engajamento                    +38%   
                        retenção                       +39%   
                        conversões por mil impressões  +48%   
rosto humano: não → sim CTR                            +23%   
                        engajamento                    +43%   
                        retenção                       +11%   
                        conversões por mil impressões  +29%   
texto: alto → médio     CTR                            +23%   
                        engajamento                     +8%   
                        retenção                       +29%   
                        conversões por mil impressões  +36%   

                                                                     Meta  \
transição               métrica                                             
hook: não → sim         CTR                             +38% [+11%, +71%]   
                        engajamento                      +39% [+5%, +84%]   
                        retenção                        +29% [+12%, +50%]   
                        conversões por mil impressões    +38% [+6%, +80%]   
rosto humano: não → sim CTR                             +12% [−11%, +40%]   
                        engajamento                    +58% [+22%, +106%]   
                        retenção                         +14% [−2%, +32%]   
                        conversões por mil impressões   +11% [−15%, +45%]   
texto: alto → médio     CTR                            +51% [+12%, +103%]   
                        engajamento                     +14% [−22%, +67%]   
                        retenção                        +40% [+13%, +75%]   
                        conversões por mil impressões  +66% [+15%, +141%]   

                                                                  TikTok  \
transição               métrica                                            
hook: não → sim         CTR                            +51% [+16%, +97%]   
                        engajamento                     +41% [+4%, +92%]   
                        retenção                       +48% [+21%, +81%]   
                        conversões por mil impressões   +45% [+8%, +93%]   
rosto humano: não → sim CTR                             +32% [+3%, +70%]   
                        engajamento                     +31% [−2%, +76%]   
                        retenção                       +12% [−10%, +39%]   
                        conversões por mil impressões   +40% [+3%, +91%]   
texto: alto → médio     CTR                            +22% [−12%, +71%]   
                        engajamento                     +6% [−27%, +53%]   
                        retenção                       +11% [−15%, +45%]   
                        conversões por mil impressões  +31% [−11%, +94%]   

                                                                  LinkedIn  \
transição               métrica                                              
hook: não → sim         CTR                            +118% [+30%, +264%]   
                        engajamento                     +32% [−29%, +142%]   
                        retenção                         +48% [+1%, +115%]   
                        conversões por mil impressões  +121% [+31%, +274%]   
rosto humano: não → sim CTR                             +27% [−30%, +132%]   
                        engajamento                     +31% [−32%, +152%]   
                        retenção                          +8% [−25%, +56%]   
                        conversões por mil impressões   +56% [−13%, +179%]   
texto: alto → médio     CTR                              +10% [−38%, +96%]   
                        engajamento                      +1% [−58%, +142%]   
                        retenção                        +68% [+14%, +148%]   
             

Variância entre plataformas igual a zero em 9 de 12 combinações


**Sem encolhimento, por falta de benefício.** Em 9 das 12 combinações, a variância entre plataformas estimada é zero: o encolhimento devolveria exatamente o efeito geral. Nas outras 3, como no CTR do hook, ela é puxada por estimativas de intervalos largos, sobretudo no LinkedIn, e calculada com só três plataformas, o que a torna instável. O engine usa o efeito geral ajustado e mostra o contexto de outra forma:

- **pelo que se aplica:** só edições que mudam algo que a campanha tem hoje;
- **pela métrica do objetivo**, que muda a ordem das recomendações;
- **pelo formato, estimado dentro de cada plataforma,** a única característica cuja comparação depende da plataforma por construção;
- **pelo suporte histórico** dos dois lados da comparação, no recorte da campanha.

Categoria, objetivo e público entram como controles do ajuste e nos recortes de suporte, mas não geram efeitos próprios por segmento: a base não sustenta essa personalização.

## 7. Diferença prevista de score

Para cada edição, a diferença entre o score previsto depois e antes, pelo modelo final. O intervalo vem de 1.000 reamostragens das 500 campanhas, refazendo todo o treino a cada uma: imputação, padronização, quais combinações de plataforma e formato viram interação, e os coeficientes, com a penalização congelada (alpha 1). Ele mede a variabilidade da estimativa preditiva, não o efeito de editar um vídeo. A coluna da EDA traz o contraste da mesma edição na regressão da Parte 1, para comparar.

In [6]:
score = score_differences(cleaned)


def points(estimate: float, low: float, high: float) -> str:
    return f"{fmt_signed(estimate, 1)} [{fmt_signed(low, 1)}, {fmt_signed(high, 1)}]"


eda_score = {}
for transition in TRANSITIONS:
    scope = data if transition.platform is None else data[data["platform"] == transition.platform]
    controls = {} if transition.platform is None else {"controls": CONTEXT_WITHIN_PLATFORM}
    fit = fit_model(scope, TARGET, log=False, extra=transition.extra, **controls)
    estimate, low, high, _ = transition_effect(fit, transition, log=False)
    eda_score[transition.key] = points(estimate, low, high)
display(
    pd.DataFrame(
        {
            "transição": score["descricao"],
            "modelo final (pontos)": [
                points(*values)
                for values in score[["diferenca", "ic_inferior", "ic_superior"]].to_numpy()
            ],
            "EDA (pontos)": [eda_score[key] for key in score["transicao"]],
        }
    ).set_index("transição")
)

,modelo final (pontos),EDA (pontos)
transição,,
hook: não → sim,"+17,0 [+15,7, +18,5]","+17,1 [+15,6, +18,7]"
rosto humano: não → sim,"+11,8 [+10,4, +13,3]","+11,8 [+10,2, +13,3]"
CTA: não → sim,"+8,1 [+6,6, +9,5]","+8,2 [+6,5, +9,8]"
legenda: não → sim,"+8,3 [+6,8, +9,8]","+8,3 [+6,7, +9,8]"
texto: alto → médio,"+14,0 [+12,0, +16,1]","+14,3 [+12,1, +16,6]"
texto: alto → baixo,"+7,2 [+5,1, +9,4]","+7,4 [+5,1, +9,8]"
texto: baixo → médio,"+6,8 [+5,3, +8,3]","+6,9 [+5,2, +8,6]"
"música/voz: +0,1 (de 0,4 para 0,5)","−0,5 [−0,9, −0,1]","−0,5 [−0,9, 0]"
duração: dobrar (de 15 para 30 s),"−0,2 [−1,3, +0,8]","−0,3 [−1,4, +0,8]"


- **O modelo é aditivo:** a diferença de uma edição é a mesma em qualquer campanha, exceto no formato, que depende da plataforma, e quando a previsão encosta nos limites de 0 ou 100. O engine vai dizer isso, em vez de apresentar o ganho como algo calculado para aquele vídeo.
- **Nas características gerais, modelo e EDA concordam,** com diferenças de até 0,3 ponto.
- **No formato, os intervalos do modelo são mais estreitos que os da EDA.** No LinkedIn, vertical → horizontal tem intervalo de cerca de +2 a +13 pontos no modelo, e de −0,1 a +18,8 na regressão dentro da plataforma. O modelo combina o efeito geral do formato, estimado com todas as plataformas, com uma interação penalizada, enquanto a EDA usa só as 83 campanhas do LinkedIn. O intervalo estreito vem dessa estrutura, não de mais evidência: são só 11 vídeos verticais no LinkedIn, e o engine sinaliza isso pelo suporte.
- **Duração e música/voz quase não mexem no score,** e o formato quadrado na Meta tem diferença compatível com zero.
- **No TikTok, quadrado → vertical é uma candidata experimental, não uma melhoria de alta confiança.** O intervalo do score vai de 0,01 a 6,7 pontos, e não há evidência robusta de associação com performance. A edição entra como C por uma margem que depende da reamostragem, com aviso de ganho limítrofe; a elegibilidade final também depende do suporte no recorte da campanha.
- **No engine, o score antes e depois vem do preditor,** para a campanha concreta e dentro dos limites de 0 a 100. Esta tabela entra como evidência complementar: o intervalo da diferença e a comparação com a EDA.

## 8. O que as evidências decidem para o engine

| Edição | Decisão | Por quê |
| :-- | :-- | :-- |
| Hook: não → sim | Entra | Associação robusta e estável em todas as métricas de objetivo |
| Rosto: não → sim | Entra | Robusta em CTR, engajamento e conversões; indício em retenção |
| Texto: alto → médio | Entra, como única recomendação de texto para quem tem texto alto | Retenção robusta. O médio supera o baixo no score (+14 contra +7) e na retenção e em conversões; o baixo só vai melhor no ROAS, de forma sensível. A alternativa aparece na justificativa, não como outra recomendação. |
| Texto: baixo → médio | Entra só como oportunidade de score (C) | Score +6,8, sem associação com performance |
| Legenda: não → sim | Entra só como C, com nota de dependência do modelo | Score +8,3; inconclusiva no log, robusta na binomial negativa. Legenda sem marcação não recebe recomendação. |
| CTA: não → sim | Entra só como C | Score +8,1, sem associação com performance |
| Formato na Meta: horizontal → vertical | Entra | Engajamento robusto (+63%); nas outras métricas, inconclusivo, e no CTR com sinal desfavorável, o que vira aviso |
| Formato no TikTok: quadrado → vertical | Só como C, candidata experimental, com aviso de ganho limítrofe | Inconclusivo em performance; o intervalo do score começa em 0,01, uma margem que depende da reamostragem. A elegibilidade final também depende do suporte. |
| Formato no LinkedIn: vertical ou quadrado → horizontal | Só como C, com aviso de extrapolação | Inconclusivo, com intervalos enormes; 11 vídeos verticais |
| Formato na Meta: quadrado → vertical | Descartada | Sem associação e sem diferença de score |
| Duração | Descartada | Nenhuma associação, com intervalos estreitos. O exemplo do enunciado, reduzir a duração no TikTok, não se sustenta nesta base. |
| Música/voz | Descartada | Retenção +3,9% a cada 0,1 de música, mas o score cai: um conflito pequeno, com efeito estimado só de forma linear |

Nenhuma edição do catálogo tem hoje associação desfavorável com p < 0,05 na métrica de algum objetivo. A regra de exclusão fica para o caso de as evidências mudarem.

## 9. Exportação e limitações

As duas tabelas vão para `reports/`, como o log de correções da Parte 1: uma linha por transição e métrica, com estimativa, intervalo, n de cada lado, p, q, classe, sensibilidades e limitações; e uma linha por edição com a diferença de score e o intervalo. O engine as lê, e um teste confere que elas batem com um recálculo.

In [7]:
save_evidence(evidence, score)
for path, table in [(PERFORMANCE_PATH, evidence), (SCORE_PATH, score)]:
    print(f"{len(table)} linhas salvas em reports/{path.name}")

70 linhas salvas em reports/recommendation_evidence.csv
14 linhas salvas em reports/score_differences.csv


**Limitações desta etapa:**

- **Associação, não causa.** Criativos com hook podem diferir em qualidade de produção ou oferta, coisas que a base não mede. Toda recomendação é uma hipótese para teste A/B.
- **Análise exploratória.** A família e os contrastes foram escolhidos depois da EDA. A correção controla a multiplicidade dentro da família, não o fato de já termos visto os dados.
- **Tetos e zeros.** Os possíveis tetos de CTR, engajamento e retenção não são modelados como censura, e as conversões dependem do modelo para as campanhas sem conversão.
- **Correções por plausibilidade.** Parte da evidência do texto depende de 5 campanhas corrigidas por plausibilidade na Parte 1.
- **Base simulada e pequena,** com 83 campanhas no LinkedIn e 11 vídeos verticais nele.